# Losses: Divergences, Information Bottleneck, and Coding Rate (C03 Topic 09)

**Citations:** Kullback & Leibler (1951); Rényi (1961); Bregman (1967); Itakura & Saito (1968); Tishby et al. (1999 VIB); Alemi et al. (2016 Deep VIB); Yu et al. (2020 MCR²).

### Purpose & Topic Bridges
Information-theoretic divergences evaluate geometric discrepancies between probability distributions, enforce compression trade-offs, and govern volumetric representation expansion. This notebook implements Forward KL (mode-covering), Reverse KL (mode-seeking), Jensen-Shannon Divergence, alpha-Renyi Divergence, scale-invariant Itakura-Saito distance, Variational Information Bottleneck (VIB), and Maximal Coding Rate Reduction (MCR²).

## 2. Mathematical Foundations & Governing Equations

1. **Forward vs. Reverse KL:** $D_{\text{KL}}(P \| Q) = \sum P \log(P/Q)$ (zero-avoiding / mode-covering) vs $D_{\text{KL}}(Q \| P) = \sum Q \log(Q/P)$ (zero-forcing / mode-seeking).
2. **Jensen-Shannon Divergence (JSD):** $\text{JSD}(P \| Q) = \frac{1}{2} D_{\text{KL}}(P \| M) + \frac{1}{2} D_{\text{KL}}(Q \| M)$, bounded in $[0, \log 2]$ with metric square root.
3. **$\alpha$-Rényi Divergence:** $D_\alpha(P \| Q) = \frac{1}{\alpha - 1} \log \sum P^\alpha Q^{1-\alpha}$, generalizing statistical divergences.
4. **Itakura-Saito Distance:** $d_{\text{IS}}(x, y) = \sum (\frac{x}{y} - \log\frac{x}{y} - 1)$, scale-invariant Bregman divergence for audio/DSP power spectra.
5. **VIB Gaussian Prior KL:** $D_{\text{KL}}(q(z|x) \| \mathcal{N}(0, I)) = -\frac{1}{2} \sum (1 + \log\sigma^2 - \mu^2 - \sigma^2)$, compressing nuisance entropy.
6. **MCR² Coding Rate:** $R(Z) = \frac{1}{2} \log \det(I + \frac{D}{N \epsilon^2} Z^T Z)$, expanding representation volume.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/09_losses_divergences_information_bottleneck_and_coding_rate.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class DivergenceUnifiedModule(nn.Module):
    def __init__(self, in_dim=8, out_dim=4):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def kl_forward(self, p, q, eps=1e-12):
        ps, qs = p.clamp(min=eps), q.clamp(min=eps)
        return (ps * (torch.log(ps) - torch.log(qs))).sum(dim=-1).mean()

    def kl_reverse(self, p, q, eps=1e-12):
        return self.kl_forward(q, p, eps)

    def jsd(self, p, q):
        m = 0.5 * (p + q)
        return 0.5 * self.kl_forward(p, m) + 0.5 * self.kl_forward(q, m)

    def renyi_divergence(self, p, q, alpha=0.5, eps=1e-12):
        ps, qs = p.clamp(min=eps), q.clamp(min=eps)
        inner = (ps.pow(alpha) * qs.pow(1.0 - alpha)).sum(dim=-1)
        return (torch.log(inner.clamp(min=eps)) / (alpha - 1.0)).mean()

    def itakura_saito(self, x, y, eps=1e-12):
        ratio = x.clamp(min=eps) / y.clamp(min=eps)
        return (ratio - torch.log(ratio) - 1.0).mean()

    def vib_kl(self, mu, logvar):
        return (-0.5 * torch.sum(1.0 + logvar - mu.pow(2) - torch.exp(logvar), dim=-1)).mean()

    def mcr2_coding_rate(self, z, eps=0.5):
        n, d = z.shape
        scale = float(d) / (float(n) * (eps ** 2))
        c = torch.eye(d, device=z.device) + scale * torch.matmul(z.t(), z)
        return 0.5 * torch.logdet(c)

model = DivergenceUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('DivergenceUnifiedModule initialized successfully.')

In [ ]:
p_d, q_d = weights['p_dist'], weights['q_dist']
r_a = float(weights['renyi_alpha'][0])
is_x, is_y = weights['is_x'], weights['is_y']
v_mu, v_lv = weights['vib_mu'], weights['vib_logvar']
m_z, m_eps = weights['mcr2_z'], float(weights['mcr2_eps'][0])

err_fwd = (model.kl_forward(p_d, q_d) - weights['loss_fwd_kl'][0]).abs().item()
assert err_fwd < 1e-5, f'Forward KL parity error: {err_fwd:.2e}'

err_rev = (model.kl_reverse(p_d, q_d) - weights['loss_rev_kl'][0]).abs().item()
assert err_rev < 1e-5, f'Reverse KL parity error: {err_rev:.2e}'

err_jsd = (model.jsd(p_d, q_d) - weights['loss_jsd'][0]).abs().item()
assert err_jsd < 1e-5, f'JSD parity error: {err_jsd:.2e}'

err_ren = (model.renyi_divergence(p_d, q_d, r_a) - weights['loss_renyi'][0]).abs().item()
assert err_ren < 1e-5, f'Renyi parity error: {err_ren:.2e}'

err_is = (model.itakura_saito(is_x, is_y) - weights['loss_is'][0]).abs().item()
assert err_is < 1e-5, f'Itakura-Saito parity error: {err_is:.2e}'

err_vib = (model.vib_kl(v_mu, v_lv) - weights['loss_vib'][0]).abs().item()
assert err_vib < 1e-5, f'VIB parity error: {err_vib:.2e}'

err_mcr2 = (model.mcr2_coding_rate(m_z, m_eps) - weights['loss_mcr2'][0]).abs().item()
assert err_mcr2 < 1e-5, f'MCR2 parity error: {err_mcr2:.2e}'

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 8 invariants verified! Fwd: {err_fwd:.2e}, Rev: {err_rev:.2e}, JSD: {err_jsd:.2e}, Ren: {err_ren:.2e}, IS: {err_is:.2e}, VIB: {err_vib:.2e}, MCR2: {err_mcr2:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'Forward KL': lambda: model.kl_forward(p_d, q_d),
    'Reverse KL': lambda: model.kl_reverse(p_d, q_d),
    'Jensen-Shannon': lambda: model.jsd(p_d, q_d),
    'Renyi Divergence': lambda: model.renyi_divergence(p_d, q_d, r_a),
    'Itakura-Saito': lambda: model.itakura_saito(is_x, is_y),
    'VIB Gaussian KL': lambda: model.vib_kl(v_mu, v_lv),
    'MCR2 Coding Rate': lambda: model.mcr2_coding_rate(m_z, m_eps),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Pathological Failure Mode: Mode-Covering Catastrophe of Forward KL on Multimodal Targets vs Mode-Seeking Reverse KL
# Target P is a separated bimodal mixture: P = 0.5 * N(-5, 1) + 0.5 * N(5, 1)
# When fitting a unimodal Gaussian Q(x; mu, sigma^2), Forward KL forces Q to cover BOTH modes,
# placing its mean at mu=0 in the zero-density valley between peaks!
x_eval = torch.linspace(-10.0, 10.0, 200, device=device)
p_target = 0.5 * torch.exp(-0.5 * (x_eval + 5.0).pow(2)) / math.sqrt(2 * math.pi) + \
           0.5 * torch.exp(-0.5 * (x_eval - 5.0).pow(2)) / math.sqrt(2 * math.pi)

# Forward KL optimal unimodal model has mean mu=0 and wide variance sigma^2 ~ 26
q_fwd_valley = torch.exp(-0.5 * (x_eval - 0.0).pow(2) / 26.0) / math.sqrt(2 * math.pi * 26.0)
valley_prob_fwd = q_fwd_valley[100].item() # x=0.0 density
print(f'[FAILURE DEMONSTRATED] Forward KL forces probability mass into zero-density valley (x=0 density: {valley_prob_fwd:.4f})')
assert valley_prob_fwd > 0.05, f'Expected mode-covering to place density in the valley, got {valley_prob_fwd}'

# Reverse KL (mode-seeking) collapses onto one mode (e.g., mu=5.0, sigma^2=1.0), zeroing out the valley!
q_rev_mode = torch.exp(-0.5 * (x_eval - 5.0).pow(2) / 1.0) / math.sqrt(2 * math.pi)
valley_prob_rev = q_rev_mode[100].item()
print(f'[STABILIZED] Reverse KL mode-seeking achieves ZERO valley density: {valley_prob_rev:.2e}')
assert valley_prob_rev < 1e-5, f'Expected mode-seeking to zero out the valley, got {valley_prob_rev:.2e}'
assert torch.allclose(model(test_in), test_out, atol=1e-5), 'Linear anchor mismatch'
print('[CORRECT] Divergences, VIB, MCR2, and linear anchor verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Asymmetric Optimization Regimes:** Forward KL is mode-covering (preventing catastrophic zero coverage in VAEs), while Reverse KL is mode-seeking (preventing unrealistic intermediate states in RL distillation and DPO).
2. **Perceptual Scale Invariance:** Itakura-Saito distance is invariant to scalar energy scaling, making it optimal for spectral audio and speech modeling across dynamic volume ranges.
3. **White-Box Subspace Packing:** MCR² replaces empirical negative pair heuristics with closed-form log-determinant volume metrics, provably maximizing subspace separation.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 08_losses_metric_contrastive_and_structured_vision](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/08_losses_metric_contrastive_and_structured_vision.ipynb)